In [ ]:
# Cell 0: GPU 確認
import subprocess, torch
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
assert torch.cuda.is_available(), '錯誤：沒有 GPU！請檢查 Runtime 設定'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')
if vram_gb < 70:
    print('WARNING: VRAM < 70GB，建議使用 A100 80GB')
else:
    print('✓ GPU 規格符合需求')

In [ ]:
# Cell 1: 使用者設定（必填）
# ============================================================
HF_USERNAME = 'Lebruhbruh'
HF_TOKEN    = ''   # <-- 貼上你的 Write Token

# 已完成標注+修正的資料集（直接用於訓練，不需要再 merge / 標注）
#   - 273 episodes / 585,425 frames
#   - 來源：AllenAI charging-01~06（共 284 eps，排除 11 個問題 eps）
#   - 標注：Pick up phone → Flip sideways → Plug cable → Turn on power
#   - 相機：top / left / right 三個視角
#   - subtask_index 0→1→2→3 在每個 episode 內單調遞增
#   - v3.0 git tag 已建立，stats.json 已重算
ANNOTATED_DATASET = 'Lebruhbruh/SARM-opendata-annotated-fixed'

# 訓練好的 SARM 模型 repo
MODEL_REPO_ID = f'{HF_USERNAME}/sarm-charging-bimanual'

# 主要訓練相機
VIDEO_KEY = 'observation.images.top'

# 四個子任務（已標注在 ANNOTATED_DATASET 中，subtask_index 0–3）
SUBTASK_NAMES = [
    'Pick up the phone',
    'Flip the phone sideways',
    'Pick up the charging cable and plug it into the phone',
    'Turn on the power of the extension cord',
]

print(f'標注資料集: {ANNOTATED_DATASET}')
print(f'SARM 模型:  {MODEL_REPO_ID}')
print(f'相機:       {VIDEO_KEY}')
print('子任務:')
for i, t in enumerate(SUBTASK_NAMES):
    print(f'  [{i}] {t}')

In [ ]:
# Cell 1b: 進度回饋工具（後續長時 cell 共用）
# ============================================================
# progress_stage(title, steps, heartbeat_seconds): context manager
#   - 開頭印 ━━━ 標題 ━━━ 與步驟清單
#   - 背景 thread 每 N 秒印一次 ⏱ 心跳（含已經過時間）
#   - 結尾印 ✓ 與總耗時
# 預期被 Cell 2 / 4b / 9a / 10 / 12 / 13 / 14 等長時 cell 使用。
# ============================================================
import threading, time, contextlib
from datetime import timedelta

def _fmt_elapsed(seconds: float) -> str:
    return str(timedelta(seconds=int(seconds)))

@contextlib.contextmanager
def progress_stage(title: str, steps=None, heartbeat_seconds: int = 30):
    print(f'\n━━━ {title} ━━━')
    if steps:
        for i, step in enumerate(steps, 1):
            print(f'  {i}) {step}')
        print()

    stop_evt = threading.Event()
    start = time.monotonic()

    def _heartbeat():
        while not stop_evt.wait(heartbeat_seconds):
            print(f'  ⏱  已執行 {_fmt_elapsed(time.monotonic() - start)}', flush=True)

    th = threading.Thread(target=_heartbeat, daemon=True)
    th.start()
    error = None
    try:
        yield
    except BaseException as e:
        error = e
        raise
    finally:
        stop_evt.set()
        th.join(timeout=1)
        total = _fmt_elapsed(time.monotonic() - start)
        if error is None:
            print(f'\n✓ {title} 完成，總耗時 {total}')
        else:
            print(f'\n✗ {title} 失敗（{type(error).__name__}），已執行 {total}')

print('✓ progress_stage 已就緒')


In [ ]:
# Cell 2: 安裝 LeRobot + SARM
import subprocess, sys

with progress_stage(
    '安裝 LeRobot + SARM（預計 2-5 分鐘）',
    steps=['pip install lerobot[sarm] from GitHub', '確認 lerobot / transformers 版本'],
    heartbeat_seconds=30,
):
    print('1/2 安裝 LeRobot + SARM...')
    # CLAUDE.md 規則 4：>30 秒指令不可 capture_output；讓 pip 即時輸出
    r = subprocess.run(
        'pip install -q "git+https://github.com/huggingface/lerobot.git#egg=lerobot[sarm]"',
        shell=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f'pip install 失敗，返回碼 {r.returncode}')
    print('  ✓ LeRobot 安裝完成')

    print('2/2 確認安裝...')
    check_cmd = 'import lerobot, transformers; print(lerobot.__version__); print(transformers.__version__)'
    r = subprocess.run([sys.executable, '-c', check_cmd], capture_output=True, text=True)
    if r.returncode == 0:
        versions = r.stdout.strip().split()
        print(f'  LeRobot: {versions[0]}')
        print(f'  Transformers: {versions[1] if len(versions)>1 else "?"}')
    else:
        raise RuntimeError(f'版本確認失敗: {r.stderr}')


In [ ]:
# Cell 3: 修補 Transformers 5.x Bug（CRITICAL）
import os, glob, torch, subprocess
import lerobot

lerobot_root = os.path.dirname(lerobot.__file__)
print(f"LeRobot 安裝路徑: {lerobot_root}")

hits = glob.glob(os.path.join(lerobot_root, "**", "processor_sarm.py"), recursive=True)
if not hits:
    print("lerobot_root 下找不到，往上一層搜尋...")
    hits = glob.glob(os.path.join(os.path.dirname(lerobot_root), "**", "processor_sarm.py"), recursive=True)

if not hits:
    raise FileNotFoundError(
        "找不到 processor_sarm.py，請確認 lerobot[sarm] 安裝成功。"
        "可在新 cell 執行: !find /usr/local/lib -name processor_sarm.py"
    )

processor_path = hits[0]
print(f"修補目標: {processor_path}")

with open(processor_path) as f:
    content = f.read()

patched = False

indent = "\n        "
old_img = "embeddings = self.clip_model.get_image_features(**inputs).detach().cpu()"
new_img = indent.join([
    "output = self.clip_model.get_image_features(**inputs)",
    "if not isinstance(output, torch.Tensor):",
    "    output = output.pooler_output",
    "embeddings = output.detach().cpu()",
])
if old_img in content:
    content = content.replace(old_img, new_img)
    print("✓ 已修補 image features")
    patched = True
else:
    print("ℹ image features 不需修補")

old_txt = "embeddings = self.clip_model.get_text_features(**inputs).detach().cpu()"
new_txt = indent.join([
    "output = self.clip_model.get_text_features(**inputs)",
    "if not isinstance(output, torch.Tensor):",
    "    output = output.pooler_output",
    "embeddings = output.detach().cpu()",
])
if old_txt in content:
    content = content.replace(old_txt, new_txt)
    print("✓ 已修補 text features")
    patched = True
else:
    print("ℹ text features 不需修補")

with open(processor_path, "w") as f:
    f.write(content)

if patched:
    r = subprocess.run(["grep", "-n", "pooler_output", processor_path],
                       capture_output=True, text=True)
    print("\n驗證修補（應看到 pooler_output）:")
    print(r.stdout)
else:
    print("\n✓ 程式碼已是最新版，無需修補")


In [ ]:
# Cell 4: HuggingFace 登入
from huggingface_hub import login
login(token=HF_TOKEN)
print('✓ HuggingFace 登入成功')

In [ ]:
# Cell 4b: 確認標注資料集狀態
# ============================================================
# 驗證 ANNOTATED_DATASET 已就緒：
#   - 存在於 HF Hub 且有 meta/info.json
#   - 273 episodes, 585,425 frames
#   - subtask_index: 0→1→2→3（4 個子任務）
#   - 三個相機 video 都完整（top/left/right）
#   - v3.0 git tag 存在（LeRobotDataset 載入需要）
# ============================================================
import json
from huggingface_hub import HfApi, hf_hub_download, list_repo_refs

api = HfApi(token=HF_TOKEN)
print(f'驗證 {ANNOTATED_DATASET} …')

info_path = hf_hub_download(ANNOTATED_DATASET, 'meta/info.json',
                             repo_type='dataset', token=HF_TOKEN)
with open(info_path) as f:
    info = json.load(f)

total_eps    = info.get('total_episodes', 0)
total_frames = info.get('total_frames', 0)
fps          = info.get('fps', '?')
print(f'  episodes: {total_eps}  frames: {total_frames:,}  fps: {fps}')
assert total_eps    == 273,    f'episodes 應為 273，實際為 {total_eps}'
assert total_frames == 585425, f'frames 應為 585425，實際為 {total_frames}'

features = info.get('features', {})
assert 'subtask_index' in features, 'subtask_index 不在 features 中！'
print(f'  subtask_index dtype: {features["subtask_index"].get("dtype", "?")}')

camera_keys = [k for k in features if k.startswith('observation.images.')]
print(f'  cameras: {camera_keys}')
assert len(camera_keys) >= 3, f'相機數量不足：{camera_keys}'

refs = list_repo_refs(ANNOTATED_DATASET, repo_type='dataset')
tags = [t.name for t in refs.tags]
assert 'v3.0' in tags, f'缺少 v3.0 git tag！現有 tags: {tags}'
print(f'  git tags: {tags}')

print('\n✓ 資料集驗證通過，可直接進行訓練')

In [ ]:
# Cell 5: 資料集預覽
import pandas as pd
from huggingface_hub import hf_hub_download
from lerobot.datasets.lerobot_dataset import LeRobotDataset

print(f'載入 {ANNOTATED_DATASET} metadata…')
dataset = LeRobotDataset(ANNOTATED_DATASET)

print(f'Episodes 總數: {dataset.num_episodes}')
print(f'Frames 總數:   {dataset.num_frames:,}')
print(f'FPS:           {dataset.fps}')
print(f'格式版本:      {dataset.meta.info.get("codebase_version", "unknown")}')

print('\nFeature Keys:')
for k in dataset.features:
    print(f'  {k}')

image_keys = [k for k in dataset.features if 'images' in k]
print(f'\n攝影機 Keys: {image_keys}')

print('\n子任務 (subtask_index → name):')
try:
    sp = hf_hub_download(ANNOTATED_DATASET, 'meta/subtasks.parquet',
                         repo_type='dataset', token=HF_TOKEN)
    df_sub = pd.read_parquet(sp)
    for _, row in df_sub.reset_index().iterrows():
        print(f'  [{row["subtask_index"]}] {row["subtask"]}')
except Exception:
    for i, t in enumerate(SUBTASK_NAMES):
        print(f'  [{i}] {t}')

In [ ]:
# Cell 5b: 全部 episode 列表
# ============================================================
# ANNOTATED_DATASET 共 273 個 episodes
# subtask_index 0→1→2→3 在每個 episode 內單調遞增
# ============================================================
from lerobot.datasets.lerobot_dataset import LeRobotDataset

ALL_EPISODES = list(range(dataset.num_episodes))
print(f'總共 {len(ALL_EPISODES)} 個 episodes')
print(f'前 10 個 episode indices: {ALL_EPISODES[:10]}')
print()

check_eps = ALL_EPISODES[:5]
ds_check  = LeRobotDataset(ANNOTATED_DATASET, episodes=check_eps)
print('subtask_index 分佈抽查（前 5 個 episodes）:')
for ep in check_eps:
    frames  = [s for s in ds_check if s['episode_index'].item() == ep]
    if not frames:
        continue
    indices = [s['subtask_index'].item() for s in frames]
    unique  = sorted(set(indices))
    print(f'  ep {ep}: subtask_index = {unique}  （{len(frames)} frames）')

In [ ]:
# Cell 6: 觀看範例 Episodes（top camera，每個 frame 標示 subtask_index）
from lerobot.datasets.lerobot_dataset import LeRobotDataset
import matplotlib.pyplot as plt
import numpy as np

PREVIEW_VIDEO_KEY = 'observation.images.top'
preview_eps = ALL_EPISODES[:3]

print(f'載入 episodes {preview_eps} 的 {PREVIEW_VIDEO_KEY} 影像…')
ds_preview = LeRobotDataset(ANNOTATED_DATASET, episodes=preview_eps)

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
for row, ep_idx in enumerate(preview_eps):
    ep_frames = [i for i, s in enumerate(ds_preview)
                 if s['episode_index'].item() == ep_idx]
    sample_indices = np.linspace(0, len(ep_frames) - 1, 5, dtype=int)
    for col, fi in enumerate(sample_indices):
        frame = ds_preview[ep_frames[fi]]
        img = frame[PREVIEW_VIDEO_KEY].numpy()
        if img.max() <= 1.0:
            img = (img * 255).astype(np.uint8)
        stage = frame['subtask_index'].item()
        axes[row][col].imshow(img.transpose(1, 2, 0) if img.ndim == 3 else img)
        axes[row][col].set_title(f'Ep {ep_idx}, stage {stage}')
        axes[row][col].axis('off')

plt.suptitle('Charging Task 預覽（top camera，數字為 subtask_index）', fontsize=14)
plt.tight_layout()
plt.savefig('/content/episode_preview.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ 預覽圖已儲存至 /content/episode_preview.png')

In [ ]:
# Cell 7: 標注資料集摘要
# ============================================================
# ANNOTATED_DATASET 的標注已在本地完成並修正：
#   - 使用 Qwen3-VL-30B + lerobot subtask_annotation 工具
#   - 修正了 subtask_index 排序 bug（字母序 → 時間序）
#   - 排除了 11 個缺少某個子任務的 episodes
#   - 確認所有 episode 的 subtask_index 0→1→2→3 單調遞增
# ============================================================
import json, pandas as pd
from huggingface_hub import hf_hub_download

try:
    sp = hf_hub_download(ANNOTATED_DATASET, 'meta/subtasks.parquet',
                         repo_type='dataset', token=HF_TOKEN)
    df_sub = pd.read_parquet(sp)
    print('subtasks.parquet:')
    print(df_sub.to_string())
except Exception as e:
    print(f'subtasks.parquet 讀取失敗: {e}')

try:
    tp_path = hf_hub_download(ANNOTATED_DATASET, 'meta/temporal_proportions_dense.json',
                               repo_type='dataset', token=HF_TOKEN)
    with open(tp_path) as f:
        tp = json.load(f)
    print('\ntemporal_proportions_dense.json:')
    for stage_name, prop in tp.items():
        print(f'  {stage_name}: {prop:.3f} ({prop*100:.1f}%)')
except Exception as e:
    print(f'temporal_proportions_dense.json 讀取失敗: {e}')

print(f'\n訓練用相機: {VIDEO_KEY}')

In [ ]:
# Cell 7b: AV1 codec 診斷 + patch extract_frame 改用 pyav
# ============================================================
# charging dataset 的影片是 AV1 編碼。Colab 預設 cv2 可能不帶 AV1 decoder，
# 導致視覺化 cell 的 extract_frame() 全部回傳 None。
# ============================================================
import os, glob, cv2

test_paths = (
    sorted(glob.glob(
        '/root/.cache/huggingface/lerobot/hub/'
        'datasets--Lebruhbruh--SARM-opendata-annotated-fixed/snapshots/*/'
        'videos/observation.images.top/chunk-000/*.mp4'
    ))
    or sorted(glob.glob(
        '/root/.cache/huggingface/hub/'
        'datasets--Lebruhbruh--SARM-opendata-annotated-fixed/snapshots/*/'
        'videos/observation.images.top/chunk-000/*.mp4'
    ))
)

cv2_can_read_av1 = False
if test_paths:
    cap = cv2.VideoCapture(test_paths[0])
    if cap.isOpened():
        ret, frame = cap.read()
        cv2_can_read_av1 = bool(ret and frame is not None)
    cap.release()
    print(f'AV1 codec 測試 ({os.path.basename(test_paths[0])}): '
          f'cv2 讀取 {"OK" if cv2_can_read_av1 else "FAIL"}')
else:
    print('⚠ 找不到已下載的 AV1 mp4（先跑過 Cell 5 確保 dataset metadata 已載入）')

if cv2_can_read_av1:
    print('✓ cv2 已支援 AV1，不需要 patch')
else:
    import lerobot
    lerobot_root = os.path.dirname(lerobot.__file__)
    hits = glob.glob(
        os.path.join(lerobot_root, '**', 'subtask_annotation.py'), recursive=True,
    ) or glob.glob(
        os.path.join(os.path.dirname(lerobot_root), '**', 'subtask_annotation.py'),
        recursive=True,
    )
    if not hits:
        print('⚠ 找不到 subtask_annotation.py（不影響訓練，只影響視覺化）')
    else:
        path = hits[0]
        print(f'修補目標: {path}')
        with open(path) as f:
            content = f.read()

        OLD = (
            'def extract_frame(video_path: Path, timestamp: float) -> np.ndarray | None:\n'
            '    """Extract a single frame from video at given timestamp."""\n'
            '    cap = cv2.VideoCapture(str(video_path))\n'
            '    if not cap.isOpened():\n'
            '        return None\n'
            '    cap.set(cv2.CAP_PROP_POS_MSEC, timestamp * 1000)\n'
            '    ret, frame = cap.read()\n'
            '    cap.release()\n'
            '    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ret else None'
        )
        NEW = (
            'def extract_frame(video_path: Path, timestamp: float) -> np.ndarray | None:\n'
            '    """Extract a single frame (patched: pyav-first, cv2-fallback)."""\n'
            '    try:\n'
            '        import av\n'
            '        with av.open(str(video_path)) as container:\n'
            '            stream = container.streams.video[0]\n'
            '            target_pts = int(timestamp / float(stream.time_base))\n'
            '            container.seek(target_pts, any_frame=False, backward=True, stream=stream)\n'
            '            last = None\n'
            '            for frame in container.decode(stream):\n'
            '                last = frame\n'
            '                if frame.pts is None:\n'
            '                    continue\n'
            '                if float(frame.pts * stream.time_base) >= timestamp:\n'
            '                    return frame.to_ndarray(format="rgb24")\n'
            '            return last.to_ndarray(format="rgb24") if last is not None else None\n'
            '    except Exception:\n'
            '        cap = cv2.VideoCapture(str(video_path))\n'
            '        if not cap.isOpened():\n'
            '            return None\n'
            '        cap.set(cv2.CAP_PROP_POS_MSEC, timestamp * 1000)\n'
            '        ret, frame = cap.read()\n'
            '        cap.release()\n'
            '        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ret else None'
        )

        if OLD in content:
            content = content.replace(OLD, NEW)
            with open(path, 'w') as f:
                f.write(content)
            print('✓ extract_frame 已 patch → pyav-first + cv2-fallback')
        elif 'pyav-first' in content:
            print('ℹ extract_frame 已是 patch 後版本，跳過')
        else:
            print('⚠ 找不到原始 extract_frame 文本（lerobot 版本可能不同）')

        if test_paths:
            import importlib, sys
            mod_name = 'lerobot.data_processing.sarm_annotations.subtask_annotation'
            if mod_name in sys.modules:
                importlib.reload(sys.modules[mod_name])
            from lerobot.data_processing.sarm_annotations.subtask_annotation import extract_frame
            frame_arr = extract_frame(test_paths[0], 0.5)
            if frame_arr is not None:
                print(f'✓ 驗證 OK：patch 後成功取到 frame shape={frame_arr.shape}')
            else:
                print('⚠ 驗證失敗：
                patch 後仍取不到 frame')

In [ ]:
# Cell 11c: 修補 dataset（⚠ 破壞性：覆寫 Hub 上的 dataset）
# ============================================================
# reexport_dataset.py 排除 11 個 episodes 後遺漏的修正：
#   (a) data/*.parquet 'index' 仍是原始 cumulative offset
#   (b) meta/episodes/*.parquet 'dataset_from/to_index' 同樣
#   (c) 'episode_index' 仍是稀疏 0..283（有 11 個缺號）—— LeRobot 期待緊密 0..272
# 修法：重編 episode_index + index + dataset_from/to_index → push → 移 v3.0 tag
# 一次性，跑成功 + Cell 12 驗證後可以刪除此 cell
# ============================================================
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download

assert HF_TOKEN, 'Cell 1 的 HF_TOKEN 是空的，先填上有 write 權限的 token'
api = HfApi(token=HF_TOKEN)

with progress_stage(
    '修補 dataset episode_index + index 欄位（預計 3-8 分鐘）',
    steps=[
        'snapshot 下載 meta + data parquet',
        '從 episodes.parquet 建立 old → new episode_index mapping',
        '重編每個 data/*.parquet 的 episode_index + index，追蹤每 ep 的 global 起訖',
        '回寫 meta/episodes/*.parquet（episode_index + dataset_from/to_index）',
        '對齊 meta/info.json 的 total_episodes / total_frames',
        'upload_folder 推回 Hub',
        '把 v3.0 tag 重新指向新 HEAD',
    ],
    heartbeat_seconds=30,
):
    # 1. snapshot（forced 避免拿到舊 cache）
    local_dir = Path(snapshot_download(
        repo_id=ANNOTATED_DATASET,
        repo_type='dataset',
        allow_patterns=['data/**', 'meta/**'],
        force_download=True,
    ))
    print(f'  下載到: {local_dir}')

    # 2. 從 episodes.parquet 建立 old_ep → new_ep mapping
    ep_files = sorted((local_dir / 'meta' / 'episodes').rglob('*.parquet'))
    if not ep_files:
        raise FileNotFoundError('找不到 meta/episodes/*.parquet')
    all_old_eps = []
    for f in ep_files:
        all_old_eps.extend(int(x) for x in pq.read_table(f, columns=['episode_index']).to_pandas()['episode_index'])
    all_old_eps = sorted(set(all_old_eps))
    old_to_new = {old: new for new, old in enumerate(all_old_eps)}
    print(f'  episodes: {len(all_old_eps)}，old max={all_old_eps[-1]}，new max={len(all_old_eps)-1}')
    if all_old_eps[-1] == len(all_old_eps) - 1:
        print('  ℹ episode_index 已經是緊密的，不需要 remap（但仍會走完流程）')

    # 3. 修 data/*.parquet：remap episode_index、重編 index、追蹤 ep_positions
    data_files = sorted((local_dir / 'data').rglob('*.parquet'))
    if not data_files:
        raise FileNotFoundError('找不到 data/*.parquet')
    print(f'  {len(data_files)} 個 data parquet')

    counter = 0
    ep_positions = {}  # new_ep_idx -> [global_first_row, global_last_row_inclusive]
    old_index_max = -1
    for f in data_files:
        df = pq.read_table(f).to_pandas()
        if 'episode_index' not in df.columns or 'frame_index' not in df.columns:
            raise ValueError(f'{f.name} 缺欄位: {df.columns.tolist()}')
        # 同 file 內按 (old_ep, frame) 排序，避免亂序
        df = df.sort_values(['episode_index', 'frame_index']).reset_index(drop=True)
        if 'index' in df.columns:
            old_index_max = max(old_index_max, int(df['index'].max()))
        # remap episode_index
        df['episode_index'] = df['episode_index'].astype('int64').map(old_to_new).astype('int64')
        # 追蹤每 new_ep 的位置
        for new_ep_idx, grp in df.groupby('episode_index', sort=True):
            first = counter + int(grp.index.min())
            last  = counter + int(grp.index.max())
            if new_ep_idx in ep_positions:
                ep_positions[new_ep_idx][0] = min(ep_positions[new_ep_idx][0], first)
                ep_positions[new_ep_idx][1] = max(ep_positions[new_ep_idx][1], last)
            else:
                ep_positions[new_ep_idx] = [first, last]
        # 重編 index
        df['index'] = range(counter, counter + len(df))
        counter += len(df)
        pq.write_table(pa.Table.from_pandas(df, preserve_index=False), f)
    total_frames = counter
    print(f'  index 重編: 舊 max={old_index_max} → 新 max={total_frames-1} (total={total_frames})')

    # 4. 回寫 meta/episodes/*.parquet：remap episode_index、更新 dataset_from/to_index
    old_to_max = -1
    for f in ep_files:
        df = pq.read_table(f).to_pandas()
        if 'dataset_to_index' in df.columns:
            old_to_max = max(old_to_max, int(df['dataset_to_index'].max()))
        # remap
        df['episode_index'] = df['episode_index'].astype('int64').map(old_to_new).astype('int64')
        # 重算 from/to
        df['dataset_from_index'] = df['episode_index'].map(lambda i: ep_positions[int(i)][0]).astype('int64')
        df['dataset_to_index']   = df['episode_index'].map(lambda i: ep_positions[int(i)][1] + 1).astype('int64')
        pq.write_table(pa.Table.from_pandas(df, preserve_index=False), f)
    new_to_max = max(v[1] + 1 for v in ep_positions.values())
    print(f'  dataset_to_index: 舊 max={old_to_max} → 新 max={new_to_max}')
    assert new_to_max == total_frames, f'mismatch: ep_end max={new_to_max} != total_frames={total_frames}'

    # 5. 對齊 info.json
    info_path = local_dir / 'meta' / 'info.json'
    info = json.load(open(info_path))
    changed = False
    if info.get('total_frames') != total_frames:
        print(f'  ⚠ info.json total_frames={info.get("total_frames")} → 改為 {total_frames}')
        info['total_frames'] = total_frames
        changed = True
    if info.get('total_episodes') != len(all_old_eps):
        print(f'  ⚠ info.json total_episodes={info.get("total_episodes")} → 改為 {len(all_old_eps)}')
        info['total_episodes'] = len(all_old_eps)
        changed = True
    if changed:
        with open(info_path, 'w') as f:
            json.dump(info, f, indent=2)
    else:
        print(f'  ✓ info.json 已對齊（total_frames={total_frames}, total_episodes={len(all_old_eps)}）')

    # 6. push
    commit = api.upload_folder(
        folder_path=str(local_dir),
        repo_id=ANNOTATED_DATASET,
        repo_type='dataset',
        allow_patterns=['data/**', 'meta/**'],
        commit_message='Fix: remap episode_index to dense 0..N-1, recompute index/dataset_from/to_index',
    )
    commit_oid = getattr(commit, 'oid', None) or getattr(commit, 'commit_url', commit)
    print(f'  ✓ pushed: {commit_oid}')

    # 7. 移動 v3.0 tag 到新 HEAD
    try:
        api.delete_tag(repo_id=ANNOTATED_DATASET, repo_type='dataset', tag='v3.0')
        print('  ✓ 舊 v3.0 tag 已刪')
    except Exception as e:
        print(f'  ℹ delete_tag 略過（可能本來就沒 tag）: {e}')
    api.create_tag(repo_id=ANNOTATED_DATASET, repo_type='dataset', tag='v3.0')
    print('  ✓ v3.0 tag 已重新指向最新 commit')

print('\n下一步：重跑 Cell 12 訓練')


In [ ]:
# Cell 12: 訓練 SARM Reward Model（預計 45-90 分鐘）
# ============================================================
# SARM 是 RewardModelConfig（不是 Policy），所以 lerobot-train 用
#   --reward_model.type=sarm
# 而不是 --policy.type=sarm。
# CLI 格式：draccus（帶 `--`，巢狀用 `.` 分隔）
# ============================================================
import subprocess, os, sys, shutil

env = os.environ.copy()
env.update({
    'HF_TOKEN': HF_TOKEN,
    'HUGGING_FACE_HUB_TOKEN': HF_TOKEN,
    'PYTHONUNBUFFERED': '1',   # 子程序 stdout/stderr 不要 buffer，立刻吐出來
})

lerobot_bin = shutil.which('lerobot-train')
lerobot_train = [lerobot_bin] if lerobot_bin else [sys.executable, '-m', 'lerobot.scripts.lerobot_train']
print(f'binary: {lerobot_train}')

# Fix #2: 避免 FileExistsError — lerobot 在 output_dir 已存在且 resume=False 時會拋錯
out_dir = 'outputs/train/sarm_charging_bimanual'
if os.path.isdir(out_dir):
    print(f'⚠ 清掉舊的 {out_dir}（避免 FileExistsError）')
    shutil.rmtree(out_dir, ignore_errors=True)

cmd = [
    *lerobot_train,
    # ── dataset ─────────────────────────────────────────────
    f'--dataset.repo_id={ANNOTATED_DATASET}',
    # ── reward model (SARM) ─────────────────────────────────
    '--reward_model.type=sarm',
    '--reward_model.annotation_mode=dense_only',
    f'--reward_model.image_key={VIDEO_KEY}',
    '--reward_model.state_key=observation.state',
    '--reward_model.n_obs_steps=8',
    '--reward_model.frame_gap=30',   # 30fps，gap=30 → 1 second context
    f'--reward_model.repo_id={MODEL_REPO_ID}',
    '--reward_model.push_to_hub=true',
    # ── top-level training ───────────────────────────────────
    f'--output_dir={out_dir}',
    '--batch_size=64',
    '--steps=5000',
    '--save_freq=2500',
    '--tolerance_s=0.001',           # 放寬 video timestamp 容差到 1ms（預設 1e-4 卡浮點漂移）
    '--wandb.enable=false',
]

print('命令:', ' '.join(cmd))

# 把所有輸出同時印到畫面跟 log 檔（Colab 重連也能事後看）
log_path = '/content/sarm_train.log'
print(f'log: {log_path}\n')

with progress_stage(
    '訓練 SARM Reward Model（5000 steps，預計 45-90 分鐘）',
    steps=[
        f'載入 {ANNOTATED_DATASET}（273 eps / 585,425 frames）',
        '初始化 SARMConfig（CLIP backbone + dense head，annotation_mode=dense_only）',
        '訓練 5000 steps（lerobot-train 即時印 step 進度）',
        f'推送模型到 HF Hub: {MODEL_REPO_ID}',
    ],
    heartbeat_seconds=30,
):
    # Fix #3 (v2): Popen + 逐行 read + Python print
    # 在 Colab/Jupyter 裡 subprocess 直接寫 FD 常被吞掉，
    # 改成 stdout=PIPE+stderr=STDOUT 一起接，再用 print() 走 Jupyter 通道，
    # 保證每一行（包含 traceback、tqdm 進度）都看得到。
    proc = subprocess.Popen(
        cmd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    with open(log_path, 'w') as logf:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='', flush=True)
            logf.write(line)
            logf.flush()
    returncode = proc.wait()

if returncode == 0:
    print(f'\n✓ 訓練完成！模型已上傳至: https://huggingface.co/{MODEL_REPO_ID}')
else:
    print(f'\n✗ 訓練失敗（返回碼: {returncode}）')
    print(f'   完整輸出已存到 {log_path}（可用 !tail -200 {log_path} 再看）')


In [ ]:
# Cell 13: 備份 Checkpoint 到 Google Drive（選擇性）
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

backup_path = '/content/drive/MyDrive/sarm_checkpoints/sarm_charging_bimanual'
src = 'outputs/train/sarm_charging_bimanual'

if not os.path.exists(src):
    print('找不到 checkpoint 目錄，請確認訓練已完成')
else:
    os.makedirs(backup_path, exist_ok=True)
    with progress_stage(
        '備份 Checkpoint 到 Google Drive',
        steps=[
            f'來源: {src}',
            f'目標: {backup_path}',
            '使用 shutil.copytree 複製（大型 checkpoint 可能需數分鐘）',
        ],
        heartbeat_seconds=30,
    ):
        shutil.copytree(src, backup_path, dirs_exist_ok=True)
    print(f'✓ Checkpoint 已備份至: {backup_path}')


In [ ]:
# Cell 14: 視覺化預測結果
import subprocess, os, sys

cmd = [
    sys.executable, '-m', 'lerobot.policies.sarm.compute_rabc_weights',
    '--dataset-repo-id', ANNOTATED_DATASET,
    '--reward-model-path', MODEL_REPO_ID,
    '--visualize-only',
    '--num-visualizations', '5',
    '--head-mode', 'dense',
    '--output-dir', '/content/sarm_predictions_viz',
]

env = os.environ.copy()
env.update({'HF_TOKEN': HF_TOKEN, 'HUGGING_FACE_HUB_TOKEN': HF_TOKEN})

with progress_stage(
    '產生 SARM 預測視覺化',
    steps=[
        f'載入訓練好的 SARM 模型: {MODEL_REPO_ID}',
        '對 5 個 episode 算 dense reward',
        '輸出 PNG 到 /content/sarm_predictions_viz',
    ],
    heartbeat_seconds=30,
):
    result = subprocess.run(cmd, env=env, text=True)

if result.returncode != 0:
    print(f'視覺化失敗（返回碼: {result.returncode}）')

In [ ]:
# Cell 14b: 顯示預測結果
import glob
from IPython.display import Image as IPyImage, display

print('SARM Reward Model 預測結果:')
pred_files = sorted(glob.glob('/content/sarm_predictions_viz/*.png'))
if not pred_files:
    print('找不到預測圖片，請確認 Cell 14 成功執行')
else:
    for path in pred_files[:5]:
        print(f'\n{__import__("os").path.basename(path)}:')
        display(IPyImage(path))
    print('\n預期結果：每個 episode 的進度曲線應單調遞增（0 → 1）')
    print('         子任務轉換處應有明顯的斜率變化')

In [ ]:
# Cell 15: 確認模型已上傳 HuggingFace Hub
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
try:
    info = api.model_info(MODEL_REPO_ID)
    print(f'✓ 模型已成功上傳！')
    print(f'  名稱: {info.id}')
    print(f'  最後更新: {info.last_modified}')
    print(f'  連結: https://huggingface.co/{MODEL_REPO_ID}')
except Exception as e:
    print(f'✗ 找不到模型: {e}')
    print('  請確認 Cell 12 的訓練步驟已成功完成')